In [1]:
import ast
import astpretty
import sys
import types
from pathlib import Path

import anaximander as nx

EXP_PATH = nx.REPO / "tests/testdata/prototypes/elementary/expressions.py"
sys.path.insert(0, EXP_PATH.parent.as_posix())

In [2]:
def module_file_to_ast_tree(file_path: str | Path):
    """Parses a Python module file and returns its AST."""
    path = Path(file_path)
    code = path.read_text()
    tree = ast.parse(code)
    return tree

In [3]:
import expressions as MODULE  # type: ignore # noqa: F401
TREE = module_file_to_ast_tree(EXP_PATH)


In [4]:
astpretty.pprint(TREE, show_offsets=False, indent=2)

Module(
  body=[
    ImportFrom(
      module='datetime',
      names=[alias(name='datetime', asname=None)],
      level=0,
    ),
    Import(
      names=[alias(name='anaximander', asname='nx')],
    ),
    Assign(
      targets=[Name(id='DT_FORMAT', ctx=Store())],
      value=Constant(value='%Y-%m-%d %H:%M:%S', kind=None),
      type_comment=None,
    ),
    ClassDef(
      name='MyModel',
      bases=[
        Attribute(
          value=Name(id='nx', ctx=Load()),
          attr='Model',
          ctx=Load(),
        ),
      ],
      keywords=[],
      body=[
        AnnAssign(
          target=Name(id='t', ctx=Store()),
          annotation=Name(id='datetime', ctx=Load()),
          value=Call(
            func=Attribute(
              value=Name(id='nx', ctx=Load()),
              attr='field',
              ctx=Load(),
            ),
            args=[],
            keywords=[
              keyword(
                arg='default',
                value=Call(
                  func

In [5]:
TREE.body

[ImportFrom(module='datetime', names=[alias(name='datetime', asname=None)], level=0),
 Import(names=[alias(name='anaximander', asname='nx')]),
 Assign(targets=[Name(id='DT_FORMAT', ctx=Store())], value=Constant(value='%Y-%m-%d %H:%M:%S', kind=None), type_comment=None),
 ClassDef(name='MyModel', bases=[Attribute(value=Name(id='nx', ctx=Load(...)), attr='Model', ctx=Load())], keywords=[], body=[AnnAssign(target=Name(id='t', ctx=Store(...)), annotation=Name(id='datetime', ctx=Load(...)), value=Call(func=Attribute(...), args=[], keywords=[keyword(...)]), simple=1)], decorator_list=[Call(func=Attribute(value=Name(...), attr='compile', ctx=Load(...)), args=[Constant(value='dataclasses', kind=None)], keywords=[])], type_params=[])]

In [ ]:
def model_definitions(tree: ast.Module, module: types.ModuleType):
    class_defs = {n.name: n for n in tree.body if isinstance(n, ast.ClassDef)}
    classes = {name: getattr(module, name) for name in class_defs}
    return {k: n for k, n in class_defs.items() if issubclass(classes[k], nx.Model)}


def model_fields(model_node: ast.ClassDef):
    rval = []
    for n in ast.walk(model_node):
        if isinstance(n, (ast.Assign, ast.AnnAssign)):
            try:
                assert n.value.func.attr == "field"
            except AttributeError:
                pass
            else:
                rval.append(n)
    return rval


def field_default(field_assign_node: ast.AnnAssign):
    keywords = getattr(field_assign_node.value, "keywords", [])
    for keyword in keywords:
        if keyword.arg == "default":
            return keyword.value

In [9]:
my_model = model_definitions(TREE, MODULE)["MyModel"]
t = model_fields(my_model)[0]
t_default_assignment = ast.unparse(field_default(t))
print(t_default_assignment)

datetime.strptime('2022-01-01 00:00:00', DT_FORMAT)
